**The Effect of Ridge on Multicollinearity**  
- **The Task:** Create or find a dataset with highly correlated features (multicollinearity).  
- **Objectives:**  
- Train a standard Linear Regression model and a Ridge Regression model.  
- Compare the magnitude of the coefficients between the two models. Observe how Ridge penalizes and shrinks the coefficients of correlated features to stabilize the model.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import StandardScaler

# 1. Load the dataset (Assuming 'Multicollinear.csv' is in your directory)
# If generating synthetically, you can use:
# np.random.seed(42)
# X1 = np.random.randn(100, 1)
# X2 = X1 + np.random.randn(100, 1) * 0.01  # Highly correlated with X1
# y = 3 * X1 + 2 * X2 + np.random.randn(100, 1)
# df = pd.DataFrame(np.hstack([X1, X2, y]), columns=['Feature_1', 'Feature_2', 'Target'])

df = pd.read_csv('Multicollinear.csv')

# Split features and target
X = df.iloc[:, :-1]
y = df.iloc[:, -1]

# Scale features (Crucial for Regularization)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split into Train/Test sets
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# 2. Train Standard Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# 3. Train Ridge Regression (alpha = 1.0 or higher to observe shrinkage)
ridge_model = Ridge(alpha=10.0)
ridge_model.fit(X_train, y_train)

# 4. Compare Coefficients
coef_comparison = pd.DataFrame({
    'Feature': X.columns if hasattr(X, 'columns') else [f'Feature_{i}' for i in range(X.shape[1])],
    'Linear Regression Coef': lr_model.coef_,
    'Ridge Regression Coef': ridge_model.coef_
})

print("=== Coefficient Comparison ===")
print(coef_comparison.to_string(index=False))

=== Coefficient Comparison ===
  Feature  Linear Regression Coef  Ridge Regression Coef
Feature_1                0.251207               0.116430
Feature_2               -0.014681               0.105476


**Feature Selection with Lasso CV**  
- **Dataset:** from sklearn.datasets import make_regression  
- **The Task:** Use a dataset with many features, some of which are irrelevant (e.g., adding pure noise columns to a housing dataset).  
- **Objectives:**  
- Use LassoCV to automatically find the best regularization strength ( / alpha) via cross-validation.  
- Examine the final coefficients. Identify which features Lasso pushed exactly to zero, effectively performing automatic feature selection.

In [2]:
# import numpy as np
# import pandas as pd
from sklearn.datasets import make_regression
from sklearn.linear_model import LassoCV
# from sklearn.preprocessing import StandardScaler

# 1. Generate a dataset with many features (some informative, some noise)
X_raw, y_raw = make_regression(n_samples=150, n_features=10, n_informative=3, noise=1.0, random_state=42)

# Create a DataFrame to keep track of feature names
feature_names = [f'Informative_{i}' for i in range(3)] + [f'Noise_{i}' for i in range(7)]
X = pd.DataFrame(X_raw, columns=feature_names)
y = pd.Series(y_raw)

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 2. Use LassoCV to automatically find the best alpha via Cross-Validation
# cv=5 performs 5-fold cross-validation
lasso_cv = LassoCV(cv=5, random_state=42)
lasso_cv.fit(X_scaled, y)

print(f"Optimal Regularization Strength (Alpha): {lasso_cv.alpha_:.4f}\n")

# 3. Examine final coefficients and identify selected features
lasso_coefs = pd.DataFrame({
    'Feature': feature_names,
    'Lasso Coefficient': lasso_cv.coef_
})

print("=== Lasso CV Feature Selection Results ===")
print(lasso_coefs.to_string(index=False))

# Identify excluded features
eliminated_features = lasso_coefs[lasso_coefs['Lasso Coefficient'] == 0]['Feature'].tolist()
selected_features = lasso_coefs[lasso_coefs['Lasso Coefficient'] != 0]['Feature'].tolist()

print("\nSelected Features:", selected_features)
print("Eliminated Features (Coefficient driven to 0):", eliminated_features)

Optimal Regularization Strength (Alpha): 0.0879

=== Lasso CV Feature Selection Results ===
      Feature  Lasso Coefficient
Informative_0          91.028615
Informative_1           0.000000
Informative_2           0.000000
      Noise_0           0.080593
      Noise_1          49.572651
      Noise_2           0.000000
      Noise_3           0.000000
      Noise_4           0.039180
      Noise_5          23.385606
      Noise_6          -0.000000

Selected Features: ['Informative_0', 'Noise_0', 'Noise_1', 'Noise_4', 'Noise_5']
Eliminated Features (Coefficient driven to 0): ['Informative_1', 'Informative_2', 'Noise_2', 'Noise_3', 'Noise_6']
